# Free-Lunch AI Demo 🍱

One YAML, one entrypoint, multiple free LLM providers with automatic fallback.

**Table of Contents**
1. [Installation & Setup](#1-installation--setup)
2. [Quick Start (Zero-Config)](#2-quick-start-zero-config)
3. [Custom YAML Menu](#3-custom-yaml-menu)
4. [Utilities](#4-utilities)

---
## 1. Installation & Setup
<a id='1-installation--setup'></a>

In [ ]:
!pip install git+https://github.com/tctsung/free-lunch-ai.git --force-reinstall -q

Set at least one API key. You can use a `.env` file (auto-loaded) or export directly:

In [ ]:
import os

# Option A: export keys directly (uncomment and fill in)
# os.environ["GROQ_API_KEY"] = "your-key"          # https://console.groq.com/keys
# os.environ["GOOGLE_API_KEY"] = "your-key"         # https://aistudio.google.com/app/api-keys
# os.environ["OPENROUTER_API_KEY"] = "your-key"     # https://openrouter.ai/settings/keys
# os.environ["OLLAMA_API_KEY"] = "your-key"         # https://ollama.com/settings/keys

# Option B: use a .env file (place in working directory)
# The Menu class loads .env automatically, no extra code needed.

---
## 2. Quick Start (Zero-Config)
<a id='2-quick-start-zero-config'></a>

No YAML needed. Built-in presets: `fast`, `think`, `agent`.

In [ ]:
from free_lunch import Menu

menu = Menu()  # loads .env automatically

### Fast — speed-optimized

In [ ]:
fast_llm = menu.fast()
response = fast_llm.invoke("Explain no free lunch theorem in one sentence.")

print(response.content)
print("Model used:", response.response_metadata["model_id"])

### Think — reasoning-optimized

In [ ]:
think_llm = menu.think()
response = think_llm.invoke("Prove that the square root of 2 is irrational.")

print("Model used:", response.response_metadata["model_id"])
print(response.content)

### Agent — built-in tools (web search, code exec)

In [ ]:
agent_llm = menu.agent()
response = agent_llm.invoke("What is the weather in NYC right now?")

print("Model used:", response.response_metadata["model_id"])
print(response.content)

---
## 3. Custom YAML Menu
<a id='3-custom-yaml-menu'></a>

Define your own profiles with full control over models, timeouts, and parameters.
List models in fallback order — the router tries them top-to-bottom.

In [ ]:
%%writefile my_menu.yaml
# Fast-response profile
fast:
  type: langchain
  timeout: 30
  global_timeout: 180
  models:
    - id: google::gemini-2.5-flash
    - id: groq::llama-3.1-8b-instant
    - id: openrouter::qwen/qwen3-4b:free

# Deep reasoning profile
story_teller:
  type: langchain
  timeout: 90
  global_timeout: 300
  models:
    - id: ollama::deepseek-v3.2:cloud
    - id: groq::openai/gpt-oss-120b
      params:
        reasoning_effort: high
    - id: openrouter::deepseek/deepseek-r1-0528:free

In [ ]:
my_menu = Menu(yaml_path="my_menu.yaml")

# Use any profile name defined in your YAML:
fast_llm = my_menu.fast()
response = fast_llm.invoke("Summarize the theory of relativity in 2 sentences.")

print("Model used:", response.response_metadata["model_id"])
print(response.content)

In [ ]:
story_llm = my_menu.story_teller()
response = story_llm.invoke("Write a short fable about a fox who learns to share.")

print("Model used:", response.response_metadata["model_id"])
print(response.content)

In [ ]:
# Clean up
import os
os.remove("my_menu.yaml")

---
## 4. Utilities
<a id='4-utilities'></a>

### `content_blocks_dict` (optional)

A convenience helper that flattens a LangChain `AIMessage` into a simple Python dict.
This is **completely optional** — standard LangChain `response.content` works as usual.

Useful when you just want a plain dict for logging, serialization, or simple scripts.

In [ ]:
from free_lunch import content_blocks_dict

# Reuse any response from above:
fast_llm = Menu().fast()
response = fast_llm.invoke("What is 2 + 2?")

# Flatten into a dict:
result = content_blocks_dict(response)
print(result)
# {"text": "4", "model_id": "google::gemini-2.5-flash"}